In [1]:
# EpiDM analysis framework notebook (Python)
# Title / notebook config
# - loads all data files in the repo
# - provides repeated-measures modeling scaffolding
# - starts with: Mixed Effects, Bayesian hierarchical, GEE, Fixed Effects

import os, re, json, glob, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

REPO_ROOT = Path(".").resolve()
print("Repo root:", REPO_ROOT)


Repo root: /Users/grace/EpiDM


In [2]:
# Install dependencies

!pip install pandas numpy scipy statsmodels pyreadr patsy scikit-learn matplotlib tqdm

# Optional Bayesian stack (can be heavy):
!pip install pymc arviz bambi

# Optional boosting + random effects:
!pip install gpboost


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.5/552.5 kB 15.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 29.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 40.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [bambi]m14/16 [arviz_plots]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 20.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [gpboost]m3/5 [optuna]


In [3]:
# File discovery & loaders

from scipy.io import loadmat
from tqdm.auto import tqdm

def discover_files(root: Path):
    patterns = [
        "**/*.csv",
        "**/*.tsv",
        "**/*.txt",
        "**/*.xlsx",
        "**/*.xls",
        "**/*.mat",
        "**/*.rda",
        "**/*.RData",
        "**/*.rds",   # might exist; harder to read
    ]
    files = []
    for p in patterns:
        files.extend(root.glob(p))
    # ignore git metadata
    files = [f for f in files if ".git" not in f.parts]
    return sorted(files)

def safe_read_csv(path: Path) -> pd.DataFrame:
    # Try common encodings + separators
    for enc in ["utf-8", "latin-1"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    # If it’s not comma-separated, try tab
    for enc in ["utf-8", "latin-1"]:
        try:
            return pd.read_csv(path, encoding=enc, sep="\t")
        except Exception:
            pass
    raise ValueError(f"Could not read CSV/TSV: {path}")

def safe_read_excel(path: Path) -> pd.DataFrame:
    return pd.read_excel(path)

def safe_read_mat(path: Path) -> dict:
    # Returns dict with numpy arrays / nested structs
    return loadmat(path, squeeze_me=True, struct_as_record=False)

def safe_read_rda(path: Path) -> dict:
    import pyreadr
    # returns dict: object_name -> pandas df (or other)
    res = pyreadr.read_r(path)
    return dict(res)

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [re.sub(r"\s+", "_", str(c).strip()) for c in df.columns]
    df.columns = [re.sub(r"[^0-9a-zA-Z_]", "", c) for c in df.columns]
    return df

files = discover_files(REPO_ROOT)
print(f"Discovered {len(files)} files")
for f in files[:25]:
    print(" -", f.relative_to(REPO_ROOT))


Discovered 226 files
 - Arousal_Neg.csv
 - Arousal_Neutral.csv
 - Confidence_df_50edit.csv
 - E_Data/10_E.csv
 - E_Data/12_E.csv
 - E_Data/14_E.csv
 - E_Data/15_E.csv
 - E_Data/16_E.csv
 - E_Data/17_E.csv
 - E_Data/19_E.csv
 - E_Data/1_E.csv
 - E_Data/20_E.csv
 - E_Data/21_E.csv
 - E_Data/23_E.csv
 - E_Data/24_E.csv
 - E_Data/27_E.csv
 - E_Data/28_E.csv
 - E_Data/29_E.csv
 - E_Data/2_E.csv
 - E_Data/30_E.csv
 - E_Data/31_E.csv
 - E_Data/33_E.csv
 - E_Data/34_E.csv
 - E_Data/35_E.csv
 - E_Data/36_E.csv


In [5]:
# Loading everything into single "data registry"

data_registry = {
    "tables": {},   # key -> DataFrame
    "mats": {},     # key -> dict
    "r_objects": {},# key -> dict of objects
    "unreadable": []
}

def key_for(path: Path) -> str:
    return str(path.relative_to(REPO_ROOT)).replace("\\", "/")

for path in tqdm(files):
    k = key_for(path)
    try:
        suffix = path.suffix.lower()
        if suffix in [".csv", ".tsv", ".txt"]:
            df = safe_read_csv(path)
            df = normalize_columns(df)
            data_registry["tables"][k] = df
        elif suffix in [".xlsx", ".xls"]:
            df = safe_read_excel(path)
            df = normalize_columns(df)
            data_registry["tables"][k] = df
        elif suffix == ".mat":
            data_registry["mats"][k] = safe_read_mat(path)
        elif suffix in [".rda", ".rdata"]:
            objs = safe_read_rda(path)
            # normalize DataFrame columns inside any DF objects
            for oname, oval in list(objs.items()):
                if isinstance(oval, pd.DataFrame):
                    objs[oname] = normalize_columns(oval)
            data_registry["r_objects"][k] = objs
        else:
            # .rds not supported here without extra work
            data_registry["unreadable"].append(k)
    except Exception as e:
        data_registry["unreadable"].append((k, str(e)))

print("Loaded tables:", len(data_registry["tables"]))
print("Loaded MAT files:", len(data_registry["mats"]))
print("Loaded RDA/RData files:", len(data_registry["r_objects"]))
print("Unreadable:", len(data_registry["unreadable"]))


100%|████████████████████████████████████████| 226/226 [00:00<00:00, 581.50it/s]

Loaded tables: 204
Loaded MAT files: 6
Loaded RDA/RData files: 0
Unreadable: 16


In [6]:
# Inventory/look at dataframes

def summarize_df(df: pd.DataFrame, n=3):
    return {
        "rows": int(df.shape[0]),
        "cols": int(df.shape[1]),
        "columns": list(df.columns[:30]),
        "head": df.head(n)
    }

# Show a compact table inventory
inv = []
for name, df in data_registry["tables"].items():
    inv.append([name, df.shape[0], df.shape[1]])
inv_df = pd.DataFrame(inv, columns=["table", "rows", "cols"]).sort_values(["rows","cols"], ascending=False)
display(inv_df.head(30))

# Peek the biggest table
if len(inv_df):
    biggest = inv_df.iloc[0]["table"]
    print("Biggest table:", biggest)
    display(data_registry["tables"][biggest].head())


,table,rows,cols
203,Value_acc_50edit.csv,7998,5
1,Arousal_Neutral.csv,282,3
0,Arousal_Neg.csv,255,3
9,E_Data/19_E.csv,252,77
10,E_Data/1_E.csv,252,77
14,E_Data/24_E.csv,252,77
16,E_Data/28_E.csv,252,77
17,E_Data/29_E.csv,252,77
23,E_Data/35_E.csv,252,77
35,E_Data/46_E.csv,252,77


Biggest table: Value_acc_50edit.csv


,ID,MemoryType,Valence,AbsDiff,Accuracy
0,1,OO,Neutral,1,0
1,1,ON,Negative,5,0
2,1,OO,Negative,4,1
3,1,ON,Neutral,3,0
4,1,OO,Neutral,5,1


In [7]:
# Helpers to detect participant/time keys

LIKELY_ID_COLS = ["subj", "subject", "participant", "pid", "id", "prolific", "worker", "user"]
LIKELY_TIME_COLS = ["time", "timestamp", "date", "datetime", "trial", "trialnum", "block", "session"]

def find_key_cols(df: pd.DataFrame):
    cols_lower = {c: c.lower() for c in df.columns}
    id_cols = [c for c in df.columns if any(tok in cols_lower[c] for tok in LIKELY_ID_COLS)]
    time_cols = [c for c in df.columns if any(tok in cols_lower[c] for tok in LIKELY_TIME_COLS)]
    return id_cols, time_cols

keys = []
for name, df in data_registry["tables"].items():
    id_cols, time_cols = find_key_cols(df)
    keys.append([name, id_cols[:5], time_cols[:5]])
keys_df = pd.DataFrame(keys, columns=["table","id_cols_guess","time_cols_guess"])
display(keys_df.head(30))


,table,id_cols_guess,time_cols_guess
0,Arousal_Neg.csv,[],[]
1,Arousal_Neutral.csv,[],[]
2,Confidence_df_50edit.csv,"[ID, Avg_Confidence]",[]
3,E_Data/10_E.csv,[participant],"[blockFile, Instructions_LoopthisTrialN, Redo_..."
4,E_Data/12_E.csv,[participant],"[blockFile, Instructions_LoopthisTrialN, Redo_..."
5,E_Data/14_E.csv,[participant],"[blockFile, Instructions_LoopthisTrialN, Redo_..."
6,E_Data/15_E.csv,[participant],"[blockFile, Instructions_LoopthisTrialN, Redo_..."
7,E_Data/16_E.csv,[participant],"[blockFile, Instructions_LoopthisTrialN, Redo_..."
8,E_Data/17_E.csv,[participant],"[blockFile, Instructions_LoopthisTrialN, Redo_..."
9,E_Data/19_E.csv,[participant],"[blockFile, Instructions_LoopthisTrialN, Redo_..."


In [8]:
# Build merge plan (work in progress)

# Pick a "base" table (usually the one with repeated-measures rows)
# Heuristic: largest number of rows
base_table_name = inv_df.iloc[0]["table"]
base = data_registry["tables"][base_table_name].copy()

print("Base table:", base_table_name, base.shape)
display(base.head())

base_id_cols, base_time_cols = find_key_cols(base)
print("Guessed ID columns:", base_id_cols)
print("Guessed time columns:", base_time_cols)

# Choose merge keys
# If there are multiple candidates, take the first; you'll edit after inspecting columns.
MERGE_KEYS = []
if base_id_cols:
    MERGE_KEYS.append(base_id_cols[0])
if base_time_cols:
    MERGE_KEYS.append(base_time_cols[0])

print("Using merge keys:", MERGE_KEYS)


Base table: Value_acc_50edit.csv (7998, 5)


,ID,MemoryType,Valence,AbsDiff,Accuracy
0,1,OO,Neutral,1,0
1,1,ON,Negative,5,0
2,1,OO,Negative,4,1
3,1,ON,Neutral,3,0
4,1,OO,Neutral,5,1


Guessed ID columns: ['ID']
Guessed time columns: []
Using merge keys: ['ID']


In [9]:
# Merge as much as possible

master = base
merged_tables = [base_table_name]
failed_merges = []

for name, df in data_registry["tables"].items():
    if name == base_table_name:
        continue
    # Only try if it shares the merge keys
    if all(k in df.columns for k in MERGE_KEYS) and all(k in master.columns for k in MERGE_KEYS):
        try:
            before = master.shape
            master = master.merge(df, on=MERGE_KEYS, how="left", suffixes=("", f"__{Path(name).stem}"))
            after = master.shape
            merged_tables.append(name)
            print(f"Merged {name}: {before} -> {after}")
        except Exception as e:
            failed_merges.append((name, str(e)))
    else:
        failed_merges.append((name, "no shared merge keys"))

print("\nMerged tables count:", len(merged_tables))
print("Failed merges count:", len(failed_merges))
display(master.head())


Merged Confidence_df_50edit.csv: (7998, 5) -> (32312, 8)
Merged Memory_perf_50.csv: (32312, 8) -> (64624, 11)
Merged Valenced_perf_50.csv: (64624, 11) -> (129248, 14)

Merged tables count: 4
Failed merges count: 200


,ID,MemoryType,Valence,AbsDiff,Accuracy,MemoryType__Confidence_df_50edit,Valence__Confidence_df_50edit,Avg_Confidence,d,c,MemoryType__Memory_perf_50,d__Valenced_perf_50,c__Valenced_perf_50,Valence__Valenced_perf_50
0,1,OO,Neutral,1,0,ON,Negative,1.600000,0.061,-0.0007,ON,-0.46,0.015,Negative
1,1,OO,Neutral,1,0,ON,Negative,1.600000,0.061,-0.0007,ON,0.22,-0.050,Neutral
2,1,OO,Neutral,1,0,ON,Negative,1.600000,0.620,-0.0240,OO,-0.46,0.015,Negative
3,1,OO,Neutral,1,0,ON,Negative,1.600000,0.620,-0.0240,OO,0.22,-0.050,Neutral
4,1,OO,Neutral,1,0,ON,Neutral,2.066667,0.061,-0.0007,ON,-0.46,0.015,Negative


In [17]:
for c in master.columns:
    print(c)


ID
MemoryType
Valence
AbsDiff
Accuracy
MemoryType__Confidence_df_50edit
Valence__Confidence_df_50edit
Avg_Confidence
d
c
MemoryType__Memory_perf_50
d__Valenced_perf_50
c__Valenced_perf_50
Valence__Valenced_perf_50


In [10]:
# Modeling scaffolds

# After inspecting `master.columns`, define:
#  - participant column
#  - time column (if relevant)
#  - outcome variable(s)
#  - predictor set

participant_col = MERGE_KEYS[0] if MERGE_KEYS else None
time_col = MERGE_KEYS[1] if len(MERGE_KEYS) > 1 else None

print("participant_col =", participant_col)
print("time_col =", time_col)

# TODO: set these after you inspect master.columns
OUTCOME = None  # e.g., "memory_correct" or "choice" or "valenced_perf"
PREDICTORS = [] # e.g., ["arousal", "confidence", "valence", "trial", ...]


participant_col = ID
time_col = None


In [11]:
# Basic cleaning utilities

def prepare_model_df(df: pd.DataFrame, outcome: str, predictors: list, group: str):
    use_cols = [outcome] + predictors + [group]
    use_cols = [c for c in use_cols if c is not None]
    d = df[use_cols].copy()

    # Drop missing outcome/group
    d = d.dropna(subset=[outcome, group])

    # Convert group to categorical
    d[group] = d[group].astype("category")

    # Try numeric conversion for predictors where possible
    for c in predictors:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="ignore")

    return d

# Example (won't run until OUTCOME/PREDICTORS are set)
# model_df = prepare_model_df(master, OUTCOME, PREDICTORS, participant_col)
# display(model_df.head())


In [13]:
# Mixed effects (LMM/GLMM!) Stat model

import statsmodels.api as sm
import statsmodels.formula.api as smf

def fit_mixedlm_continuous(df: pd.DataFrame, outcome: str, predictors: list, group: str):
    # LMM for continuous outcome
    # Formula: outcome ~ x1 + x2 + ...
    rhs = " + ".join(predictors) if predictors else "1"
    formula = f"{outcome} ~ {rhs}"
    model = smf.mixedlm(formula, df, groups=df[group])
    res = model.fit(reml=False, method="lbfgs")
    return res

# If binary outcome, prefer GLMM (not fully supported in vanilla statsmodels MixedLM).
# Workarounds: use Bayesian (below), or use GEE for population-average, or use R (lme4/glmer).


In [14]:
# Hierarchical Bayesian

# This is the cleanest way in Python to do a GLMM-style model.

# Uncomment once installed:
# import bambi as bmb
# import arviz as az

def fit_bayesian_hierarchical(df: pd.DataFrame, outcome: str, predictors: list, group: str, family="gaussian"):
    """
    family:
      - "gaussian" for continuous
      - "bernoulli" for binary
      - "categorical" for multiclass
      - "ordinal" for ordinal (with appropriate setup)
    """
    rhs = " + ".join(predictors) if predictors else "1"
    # random intercept
    formula = f"{outcome} ~ {rhs} + (1|{group})"
    # For random slopes, add: + (1 + x1|group)
    model = bmb.Model(formula, df, family=family)
    idata = model.fit(draws=1500, tune=1500, target_accept=0.9)
    return model, idata

# Example:
# model_df = prepare_model_df(master, OUTCOME, PREDICTORS, participant_col)
# model, idata = fit_bayesian_hierarchical(model_df, OUTCOME, PREDICTORS, participant_col, family="bernoulli")
# az.summary(idata)


In [15]:
# GEE (population-average, robust SEs

from statsmodels.genmod.generalized_estimating_equations import GEE
from statsmodels.genmod.families import Gaussian, Binomial
from statsmodels.genmod.cov_struct import Exchangeable, Autoregressive

def fit_gee(df: pd.DataFrame, outcome: str, predictors: list, group: str, family="gaussian", corr="exchangeable"):
    rhs = " + ".join(predictors) if predictors else "1"
    formula = f"{outcome} ~ {rhs}"

    fam = Gaussian() if family == "gaussian" else Binomial()

    cov = Exchangeable() if corr == "exchangeable" else Autoregressive()

    model = smf.gee(formula, groups=df[group], data=df, family=fam, cov_struct=cov)
    res = model.fit()
    return res


In [16]:
# Fixed effects regression

def fit_fixed_effects(df: pd.DataFrame, outcome: str, predictors: list, group: str):
    # Adds participant dummies via C(group)
    rhs = " + ".join(predictors) if predictors else "1"
    formula = f"{outcome} ~ {rhs} + C({group})"
    res = smf.ols(formula, df).fit()
    return res
